In [74]:
import requests
from datetime import datetime
import pandas as pd
import numpy as np

In [155]:
def get_events_near_location(
    lat: float,
    lon: float,
    start_date: str,
    end_date: str,
    distance_meters: int,
    api_key: str,
):
    """
    Fetch events from PredictHQ near a location within a radius and date range.

    Parameters:
    - lat, lon: location coordinates
    - start_date, end_date: 'YYYY-MM-DD'
    - distance_meters: radius in meters
    - api_key: PredictHQ API key
    """

    # Convert meters → km (PredictHQ uses km)
    # radius_km = distance_meters / 1000

    url = "https://api.predicthq.com/v1/events/"

    headers = {"Authorization": f"Bearer {api_key}", "Accept": "application/json"}

    params = {
        "within": f"{distance_meters}m@{lat},{lon}",
        "active.gte": start_date,
        "active.lte": end_date,
        "category": "concerts,sports,festivals,performing-arts",
        "rank.gte": 40,
        "phq_attendance.gte": 200,
        # "location_confidence_score.gte": 3,
        # "sort": "-rank",
        "limit": 100,
    }

    response = requests.get(url, headers=headers, params=params)

    if response.status_code != 200:
        raise Exception(f"API Error {response.status_code}: {response.text}")

    data = response.json()

    # Return only useful event list
    return data.get("results", [])

In [102]:
events_data_path = "../../../data/France/processed_data/events_batch2.csv"
events_df = pd.read_csv(events_data_path, low_memory=False)

In [103]:
events_df.head()

,shop_lat,shop_lon,query_date_from,query_date_to,query_radius_m,source,id,name,date,time,...,price_max,price_currency,url,recurrent_event,recurrence_type,venue_type,estimated_capacity,capacity_range_min,capacity_range_max,capacity_confidence
0,48.8681,2.3524,2026-01-01,2026-04-30,1000,OpenAgenda,94029950,Les enfants veulent la paix,2026-04-11,15:30,...,NaN,EUR,https://openagenda.com/agendas/52371692/events...,False,none,indoor theater,2250.0,1500.0,3000.0,medium
1,48.8681,2.3524,2026-01-01,2026-04-30,1000,OpenAgenda,38556445,Concert “Gospel spirit”,2026-04-11,20:30,...,NaN,EUR,https://openagenda.com/agendas/86173418/events...,False,none,church,1500.0,500.0,2000.0,medium
2,48.8681,2.3524,2026-01-01,2026-04-30,1000,OpenAgenda,3127479,Concert de l'Ensemble Latitudes,2026-04-18,15:00,...,NaN,EUR,https://openagenda.com/agendas/67709699/events...,False,none,church,350.0,200.0,500.0,low
3,48.8681,2.3524,2026-01-01,2026-04-30,1000,OpenAgenda,88319171,Vigile pascale,2026-04-04,21:00,...,NaN,EUR,https://openagenda.com/agendas/67709699/events...,False,none,church,350.0,200.0,500.0,low
4,48.8681,2.3524,2026-01-01,2026-04-30,1000,OpenAgenda,96233096,Office de la Passion,2026-04-03,19:30,...,NaN,EUR,https://openagenda.com/agendas/67709699/events...,False,none,church,350.0,200.0,500.0,low


In [104]:
events_df.groupby(["shop_lat", "shop_lon"]).count().sort_values("id", ascending=False)

,,query_date_from,query_date_to,query_radius_m,source,id,name,date,time,end_time,venue,...,price_max,price_currency,url,recurrent_event,recurrence_type,venue_type,estimated_capacity,capacity_range_min,capacity_range_max,capacity_confidence
shop_lat,shop_lon,,,,,,,,,,,,,,,,,,,,,
48.8524,2.3395,191,191,191,191,191,191,191,191,191,191,...,0,191,191,191,191,191,191,191,191,191
48.8550,2.3459,190,190,190,190,190,190,190,190,190,190,...,0,190,190,190,190,190,190,190,190,190
48.8567,2.3552,185,185,185,185,185,185,185,185,185,185,...,0,185,185,185,185,185,185,185,185,185
48.8557,2.3571,183,183,183,183,183,183,183,183,183,183,...,0,183,183,183,183,183,183,183,183,183
48.8560,2.3582,181,181,181,181,181,181,181,181,181,181,...,0,181,181,181,181,181,181,181,181,181
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48.8584,2.4101,2,2,2,2,2,2,2,2,2,2,...,0,2,2,2,2,2,2,2,2,2
48.8265,2.3730,2,2,2,2,2,2,2,2,2,2,...,0,2,2,2,2,2,2,2,2,2
48.8238,2.3768,1,1,1,1,1,1,1,1,1,1,...,0,1,1,1,1,1,1,1,1,1


In [105]:
API_KEY = "sU9J8MTwRbWXGtUYp8fJ-OMI1xLty8wnPS2CdIEn"

In [156]:
events = get_events_near_location(
    lat=48.87035,
    lon=2.35998,
    start_date="2026-02-05",
    end_date="2026-03-28",
    distance_meters=300,
    api_key=API_KEY,
)

print(len(events))

13


In [157]:
import json

for event in events:
    print(event["start"])

2026-03-27T19:00:00Z
2026-03-20T19:00:00Z
2026-03-19T19:00:00Z
2026-03-11T19:00:00Z
2026-03-10T19:00:00Z
2026-02-26T19:00:00Z
2026-02-26T11:00:00Z
2026-02-21T22:00:00Z
2026-02-16T18:00:00Z
2026-02-15T19:00:00Z
2026-02-14T19:00:00Z
2026-02-12T19:00:00Z
2026-02-11T19:00:00Z


In [158]:
print(json.dumps(events[-9], indent=4))

{
    "relevance": 0.0,
    "id": "H4C8ki9rSmspkKEdup",
    "title": "Drake Milligan",
    "alternate_titles": [
        "Drake Milligan at Alhambra Paris"
    ],
    "description": "Sourced from predicthq.com",
    "category": "concerts",
    "rank": 49,
    "local_rank": 47,
    "phq_attendance": 845,
    "entities": [
        {
            "entity_id": "yCEpku22ArJxD8xdSHngaE",
            "name": "Drake Milligan",
            "type": "person"
        },
        {
            "entity_id": "XHLVmrzzaFnCNiXQ7gwBQT",
            "name": "L'Alhambra",
            "type": "venue",
            "formatted_address": "21 Rue Yves Toudic, 75010 Paris, France"
        }
    ],
    "duration": 0,
    "start": "2026-03-10T19:00:00Z",
    "start_local": "2026-03-10T20:00:00",
    "end": "2026-03-10T19:00:00Z",
    "end_local": "2026-03-10T20:00:00",
    "predicted_end": "2026-03-10T22:10:00Z",
    "predicted_end_local": "2026-03-10T23:10:00",
    "updated": "2026-03-24T19:03:39Z",
    "first_seen

In [150]:
events = get_events_near_location(
    lat=48.87684,
    lon=2.37197,
    start_date="2026-01-20",
    end_date="2026-01-30",
    distance_meters=300,
    api_key=API_KEY,
)

print(len(events))

1


In [151]:
for event in events:
    print(event["start"])

2026-01-22T19:00:00Z


In [152]:
events_df[(events_df["shop_lat"] == 48.87684) & (events_df["shop_lon"] == 2.37197)]

,shop_lat,shop_lon,query_date_from,query_date_to,query_radius_m,source,id,name,date,time,...,price_max,price_currency,url,recurrent_event,recurrence_type,venue_type,estimated_capacity,capacity_range_min,capacity_range_max,capacity_confidence
